# MNIST con PyTorch

Arquitectura común: **784 → 128 (ReLU) → 64 (ReLU) → 10 logits**. Adam (`lr=0.001`), entropía cruzada, batch size 64, 10 épocas y semilla 42.

In [ ]:
from pathlib import Path
import json
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd().resolve()
if ROOT.name == 'src':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from mnist_loader import load_mnist

SEED = 42
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 0.001
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
X_train, y_train, X_test, y_test = load_mnist(ROOT / 'data')
assert X_train.shape == (60000, 784)
assert y_train.shape == (60000,)
assert X_test.shape == (10000, 784)
assert y_test.shape == (10000,)
assert X_train.flags.writeable and y_train.flags.writeable

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for image, label, axis in zip(X_train[:10], y_train[:10], axes.flat):
    axis.imshow(image.reshape(28, 28), cmap='gray')
    axis.set_title(f'Etiqueta: {label}')
    axis.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
X_train = X_train.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0
y_train = y_train.astype(np.int64)
y_test = y_test.astype(np.int64)

train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10),
)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
model

In [ ]:
loss_curve = []
start = time.perf_counter()
model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    loss_curve.append(epoch_loss)
    print(f'Época {epoch + 1:02d}/{EPOCHS}: loss={epoch_loss:.4f}')
training_time = time.perf_counter() - start

In [ ]:
model.eval()
predictions = []
correct = 0
with torch.no_grad():
    for images, labels in test_loader:
        predicted = model(images).argmax(dim=1)
        predictions.append(predicted.numpy())
        correct += (predicted == labels).sum().item()
y_pred = np.concatenate(predictions)
accuracy = correct / len(test_loader.dataset)
cm = confusion_matrix(y_test, y_pred)
print(f'Accuracy test: {accuracy:.4%}')
print(f'Tiempo de entrenamiento: {training_time:.2f} s')
assert accuracy > 0.90, 'Accuracy inesperadamente baja; revisar el preprocesamiento o entrenamiento.'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(1, EPOCHS + 1), loss_curve, marker='o')
axes[0].set(title='Pérdida — PyTorch', xlabel='Época', ylabel='Cross-entropy')
axes[0].grid(alpha=0.3)
ConfusionMatrixDisplay(cm).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Matriz de confusión — PyTorch')
plt.tight_layout()
plt.show()

results_dir = ROOT / 'results'
results_dir.mkdir(exist_ok=True)
result = {'framework': 'PyTorch', 'accuracy': accuracy, 'training_time_seconds': training_time, 'loss': loss_curve, 'confusion_matrix': cm.tolist()}
with (results_dir / 'pytorch.json').open('w', encoding='utf-8') as file:
    json.dump(result, file, indent=2)